In [7]:
# Import packages
import torch
import torch.nn as nn
import numpy as np
from pathlib import Path

# Pathways 
out_dir = Path("..") / "plots" / "eda"
out_dir.mkdir(parents=True, exist_ok=True)
# out_path = out_dir / "heston.pdf"

In [8]:
""" Global parameters """

# Answer to the universe and everything
torch.manual_seed(42)

In [9]:
""" Black-Scholes PINN Class """

class BSPINN(nn.Module):
    def __init__(self, hidden_layers=3, neurons_per_layer=64):
        super(BSPINN, self).__init__()

        # Input layer: Takes (S, tau) -> 2 dimensions
        layers = [nn.Linear(2, neurons_per_layer), nn.Softplus()]

        # Hidden layers
        # Softplus or Tanh here since ReLU has a 2nd derivative of 0,
        # which would make V_SS completely vanish in our PDE
        for _ in range(hidden_layers):
            layers.append(nn.Linear(neurons_per_layer, neurons_per_layer))
            layers.append(nn.Softplus())

        # Outer layer: Returns V (Option Price) -> 1 dimension
        layers.append(nn.Linear(neurons_per_layer, 1))

        self.net = nn.Sequential(*layers)
    
    def forward(self, S, tau):
        # concatenate S and tau into a single input tensor
        x = torch.cat([S, tau], dim=1)
        return self.net(x)


In [10]:
""" PDE Loss """

# Function computes the Black-Scholes PDE residual
# using Automatic Differentiation
def pde_loss(model, S, tau, r, sigma):
    
    # (1) Forward pass to get option price V
    V = model(S, tau)

    # (2) First derivative dV/dS and dV/dtau
    # Make create_graph=True is critical so we can take 
    # the derivative of the derivative
    V_S = torch.autograd.grad(
        V, S,
        grad_outputs = torch.ones_like(V),
        create_graph = True
    )[0]

    V_tau = torch.autograd.grad(
        V, tau,
        grad_outputs = torch.ones_like(V),
        create_graph = True
    )[0]

    # (3) Second derivative d^2V/dS^2
    V_SS = torch.autograd.grad(
        V_S, S, 
        grad_outputs = torch.ones_like(V_S),
        create_graph = True
    )

    # (4) Construct the Black-Scholes residual
    pde_residual = V_tau - (0.5 * sigma**2 * S**2 * V_SS + r * S * V_S - r * V)

    # L_pde is the Mean Squared Error of the residual (want this to be 0)
    loss_pde = torch.mean(pde_residual**2)
    
    # Returning the PDE Loss
    return loss_pde

In [11]:
""" Total Loss """

# Formula calculating total loss
def total_loss(model, S_interior, tau_interior, S_boundary, tau_boundary, K, r, sigma):

    # (1) PDE Loss (L_pde)
    loss_pde = pde_loss(model, S_interior, tau_interior, r, sigma)

    # (2) Initial Condition Loss (L_ic)
    # At tau = 0 (maturity), the option price MUST equal the payoff max(S-K, 0)
    tau_zero = torch.zeros_like(S_interior)
    V_ic_pred = model(S_interior, tau_zero)

    # ReLU is here max(S-K, 0)
    V_ic_true = torch.relu(S_interior - K)
    
    # Finding the initial condition loss
    loss_ic = torch.mean((V_ic_pred - V_ic_true)**2)

    # (3) Boundary Condition Loss (L_bc)
    # E.g. at S = 0, a call option is worth 0
    S_zero = torch.zeros_like(tau_interior)
    V_bc_pred = model(S_zero, tau_interior)
    loss_bc = torch.mean((V_bc_pred - 0.0)**2)

    # Note: You'd also add a boundary for S -> infinity (V ≈ S - K * exp(-r*tau))

    # Total Loss
    total_loss = loss_pde + loss_ic + loss_bc
    
    # Returning total loss
    return total_loss
